<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week07-lcr-afd-dispersion-ofdm.ipynb)

# Week 7 — Level Crossing Rate, Average Fade Duration, Dispersion and OFDM
**ECE 512 Wireless Communications** · companion notebook to the Week 7 lecture

This notebook lets you experiment with the ideas from the slides:

1. Second-order fading statistics: what LCR and AFD measure
2. Level crossing rate (Rayleigh and Ricean)
3. Average fade duration
4. Multipath channel dispersion parameters: $\bar\tau$, $\sigma_\tau$, $B_c$, $T_c$
5. Fading channel classification (flat / frequency selective, slow / fast)
6. Modulation performance in fading channels
7. OFDM: cyclic prefix, one-tap equalization and pilot-based channel estimation

Run the setup cell first. In Colab: *Runtime → Run all*.

In [ ]:
# --- Setup: installs the course package in Google Colab (safe to re-run) ---
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
import importlib.util
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, FloatLogSlider, Dropdown, Checkbox
from ece512 import fading, fadingstats as fs, modulation as mod, ofdm
from ece512.units import db_to_linear, linear_to_db

## 1. Second-order fading statistics
The Rayleigh/Ricean PDF (Week 5) tells us *how often* the envelope $r=|h|$ is below a level $R$, but not *how the
fades are arranged in time*. Two second-order statistics answer that:

* **Level crossing rate** $\mathrm{LCR}(R)$ — the rate (crossings/s) at which $r(t)$ crosses $R$ in the positive direction;
* **Average fade duration** $\mathrm{AFD}(R)$ — the average time $r(t)$ stays below $R$.

Both depend on the mobile speed through the maximum Doppler frequency $f_m = v/\lambda$. We express the level as
$\rho = R/R_{\mathrm{rms}} = R/\sqrt{\Omega_p}$ with $\Omega_p = \mathrm{E}[r^2]$.

Below: a Rayleigh envelope (Clarke sum-of-sinusoids, `fading.rayleigh_sos`) with every positive-going crossing
of $\rho=-10$ dB marked and every fade shaded.

In [ ]:
fm = 50.0                                 # e.g. 60 km/h at 900 MHz
fs_hz = 100 * fm                          # 100 samples per 1/f_m
t = np.arange(int(0.4 * fs_hz)) / fs_hz   # 0.4 s = 20/f_m
r = fs.normalized_envelope(fading.rayleigh_sos(fm, t, rng=7))
rho_dB = -10.0
rho = db_to_linear(rho_dB / 2)            # amplitude ratio: 20 log10(rho) = -10 dB

up = np.flatnonzero((r[:-1] < rho) & (r[1:] >= rho)) + 1
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(1e3 * t, 20 * np.log10(r), lw=1.2, label=r"envelope $20\log_{10}(r/R_{\mathrm{rms}})$")
ax.axhline(rho_dB, color="C3", ls="--", lw=1.2, label=rf"level $\rho$ = {rho_dB:g} dB")
ax.fill_between(1e3 * t, -40, 10, where=r < rho, color="C3", alpha=0.15, lw=0, label="in a fade")
ax.plot(1e3 * t[up], np.full(up.size, rho_dB), "^", color="C2", ms=8, label="positive-going crossing")
ax.set_ylim(-35, 8); ax.set_xlabel("time t (ms)"); ax.set_ylabel("envelope (dB re $R_{\\mathrm{rms}}$)")
ax.set_title(rf"Rayleigh envelope, $f_m$ = {fm:g} Hz")
ax.grid(alpha=0.3); ax.legend(loc="lower right", fontsize=8, ncol=2); plt.show()

T = t[-1] - t[0]
d = fs.fade_durations(r, rho, fs_hz)
print(f"crossings in {1e3*T:.0f} ms: {up.size}  ->  measured LCR = {up.size/T:5.1f} /s,"
      f"  theory = {fs.lcr_rayleigh(rho, fm):5.1f} /s")
print(f"mean fade duration     : measured = {1e3*d.mean():5.2f} ms,  theory AFD = {1e3*fs.afd_rayleigh(rho, fm):5.2f} ms")
print("(a 0.4 s record is short: the estimates are noisy -- sections 2-3 use much longer records)")

## 2. Level crossing rate
The LCR follows from the joint density of envelope level and slope,
$\mathrm{LCR}(R) = \int_0^\infty \dot\alpha\, p(R,\dot\alpha)\, d\dot\alpha$. With 2-D isotropic scattering and the LOS
component at zero Doppler offset ($f_s=f_c$), level and slope are independent and

$$\mathrm{LCR}(R) = \sqrt{2\pi(K+1)}\, f_m\, \rho\, \exp\!\left(-K-(K+1)\rho^2\right) I_0\!\left(2\rho\sqrt{K(K+1)}\right),
\qquad \text{Rayleigh } (K=0):\ \ \mathrm{LCR}(R) = \sqrt{2\pi}\, f_m\, \rho\, e^{-\rho^2}.$$

The LCR scales with $f_m$, so we plot $\mathrm{LCR}/f_m$ (crossings per wavelength travelled). The Rayleigh curve peaks at
$\rho = 1/\sqrt2$ ($-3$ dB). Markers: crossings counted on long simulated envelopes (`fading.rician_sos`,
LOS at $90^\circ$ to the direction of travel so that its Doppler shift is zero).

In [ ]:
def simulate_envelope(fm, K=0.0, los_angle=np.pi / 2, n_real=10, n_samp=20000, seed=0):
    # Concatenate n_real independent envelopes, each 200/f_m long at 100 f_m samples/s
    fs_hz = 100 * fm
    t = np.arange(n_samp) / fs_hz
    r = np.concatenate([fs.normalized_envelope(fading.rician_sos(fm, t, K, los_angle=los_angle, rng=seed + s))
                        for s in range(n_real)])
    return r, fs_hz

rho_dB_grid = np.linspace(-30, 8, 200)
rho_grid = db_to_linear(rho_dB_grid / 2)
rho_meas_dB = np.arange(-30, 9, 2.0)
rho_meas = db_to_linear(rho_meas_dB / 2)

fm = 50.0
sims = {K: simulate_envelope(fm, K) for K in (0, 3, 10)}
fig, ax = plt.subplots(figsize=(7.5, 4.3))
for i, (K, (r, fs_hz)) in enumerate(sims.items()):
    ax.semilogy(rho_dB_grid, fs.lcr_rician(rho_grid, fm, K) / fm, color=f"C{i}", label=f"theory, K = {K}")
    ax.semilogy(rho_meas_dB, fs.measured_lcr(r, rho_meas, fs_hz) / fm, "o", color=f"C{i}", ms=5, mfc="none",
                label=f"simulation, K = {K}")
ax.set_ylim(1e-2, 5); ax.set_xlabel(r"normalized level $\rho = R/R_{\mathrm{rms}}$ (dB)")
ax.set_ylabel(r"$\mathrm{LCR}/f_m$ (crossings per $1/f_m$)")
ax.set_title("Level crossing rate: simulation vs. theory")
ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8, ncol=2); plt.show()

Notice that a strong LOS component ($K=10$) makes deep fades *rare* (few crossings far below $R_{\mathrm{rms}}$) but
crossings close to $R_{\mathrm{rms}}$ *more frequent*: the envelope wiggles around the LOS amplitude.

## 3. Average fade duration
Over a long interval $T$ there are $T\cdot\mathrm{LCR}(R)$ fades, and the total time in fade is $T\Pr(r\le R)$, so

$$\mathrm{AFD}(R) = \frac{\Pr(r\le R)}{\mathrm{LCR}(R)},\qquad
\Pr(r\le R) = 1 - Q\!\left(\sqrt{2K},\ \sqrt{2(K+1)}\,\rho\right)\ \ \text{(Marcum } Q\text{)},\qquad
\text{Rayleigh: }\ \mathrm{AFD}(R) = \frac{e^{\rho^2}-1}{\rho\, f_m\sqrt{2\pi}}.$$

The simulation below measures the duration of every individual fade (crossing instants interpolated between
samples) and averages them — an independent check of the ratio formula. The histogram shows that individual fade
durations are far from constant: many short fades and a long tail.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
for i, (K, (r, fs_hz)) in enumerate(sims.items()):
    ax1.semilogy(rho_dB_grid, fm * fs.afd_rician(rho_grid, fm, K), color=f"C{i}", label=f"theory, K = {K}")
    ax1.semilogy(rho_meas_dB, fm * fs.measured_afd(r, rho_meas, fs_hz), "o", color=f"C{i}", ms=5, mfc="none",
                 label=f"simulation, K = {K}")
ax1.set_ylim(5e-3, 20); ax1.set_xlabel(r"normalized level $\rho$ (dB)")
ax1.set_ylabel(r"$f_m\cdot\mathrm{AFD}$ (fade length in units of $1/f_m$)")
ax1.set_title("Average fade duration: simulation vs. theory")
ax1.grid(True, which="both", alpha=0.3); ax1.legend(fontsize=8, ncol=2)

r, fs_hz = sims[0]
rho = db_to_linear(-10 / 2)
d = fs.fade_durations(r, rho, fs_hz) * fm
ax2.hist(d, bins=40, color="C0", alpha=0.7)
ax2.axvline(d.mean(), color="C3", lw=2, label=f"measured mean = {d.mean():.3f}/$f_m$")
ax2.axvline(fm * fs.afd_rayleigh(rho, fm), color="k", ls="--", label=f"theory AFD = {fm*fs.afd_rayleigh(rho, fm):.3f}/$f_m$")
ax2.set_xlabel(r"fade duration $t_i\cdot f_m$"); ax2.set_ylabel("number of fades")
ax2.set_title(rf"Rayleigh, $\rho$ = -10 dB: {d.size} fades")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Interactive: LCR and AFD in physical units.** Change $f_m$ (speed) and $K$. Unticking *LOS at zero Doppler*
points the LOS path along the direction of travel (Doppler shift $f_m$): the Doppler spectrum is then no longer
symmetric about $f_c$ and the closed-form Ricean LCR (which assumed $f_s = f_c$) no longer applies — watch the
markers leave the curves. (Deep levels see only a few dozen crossings in the 3200/$f_m$-long record, so expect
some scatter there.)

In [ ]:
def show_lcr_afd(fm_hz=50.0, K=3.0, los_zero_doppler=True):
    r, fs_hz = simulate_envelope(fm_hz, K, los_angle=np.pi / 2 if los_zero_doppler else 0.0, n_real=16, seed=11)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.semilogy(rho_dB_grid, fs.lcr_rician(rho_grid, fm_hz, K), "C0", label="theory")
    a1.semilogy(rho_meas_dB, fs.measured_lcr(r, rho_meas, fs_hz), "o", color="C1", mfc="none", label="simulation")
    a1.set_ylabel("LCR (crossings/s)"); a1.set_ylim(fm_hz * 1e-2, fm_hz * 5)
    a2.semilogy(rho_dB_grid, 1e3 * fs.afd_rician(rho_grid, fm_hz, K), "C0", label="theory")
    a2.semilogy(rho_meas_dB, 1e3 * fs.measured_afd(r, rho_meas, fs_hz), "o", color="C1", mfc="none", label="simulation")
    a2.set_ylabel("AFD (ms)"); a2.set_ylim(5e3 / fm_hz * 1e-3, 2e4 / fm_hz)
    for a in (a1, a2):
        a.set_xlabel(r"normalized level $\rho$ (dB)"); a.grid(True, which="both", alpha=0.3); a.legend()
    los = "LOS at zero Doppler" if los_zero_doppler else r"LOS Doppler = $+f_m$"
    fig.suptitle(rf"$f_m$ = {fm_hz:g} Hz, K = {K:g}, {los}")
    plt.tight_layout(); plt.show()

interact(show_lcr_afd, fm_hz=FloatSlider(value=50, min=5, max=200, step=5, description="f_m (Hz)"),
         K=FloatSlider(value=3, min=0, max=20, step=0.5, description="K"),
         los_zero_doppler=Checkbox(value=True, description="LOS at zero Doppler"));

## 4. Multipath channel dispersion parameters
**Time dispersion.** From a power delay profile with path powers $C_k^2$ at excess delays $\tau_k$:

$$\bar\tau = \frac{\sum_k C_k^2\tau_k}{\sum_k C_k^2},\qquad
\overline{\tau^2} = \frac{\sum_k C_k^2\tau_k^2}{\sum_k C_k^2},\qquad
\sigma_\tau = \sqrt{\overline{\tau^2}-\bar\tau^{\,2}},\qquad B_c\propto\frac{1}{\sigma_\tau}.$$

The proportionality constant depends on the correlation you demand; common rules of thumb (Rappaport) are
$B_c\approx 1/(50\sigma_\tau)$ for correlation $\ge 0.9$ and $B_c\approx 1/(5\sigma_\tau)$ for correlation
$\ge 0.5$.

**Frequency dispersion.** Motion spreads the spectrum over $f_c\pm f_m$; the channel stays roughly constant for a
coherence time $T_c\propto 1/f_m$ — e.g. $T_c \approx 9/(16\pi f_m)$ (correlation $0.5$) or the geometric-mean
rule $T_c\approx 0.423/f_m$.

First, Rappaport's Example 5.4 profile and two ITU profiles:

In [ ]:
pdps = {  # name: (delays in s, powers in dB)
    "Rappaport Ex. 5.4": (np.array([0, 1, 2, 5]) * 1e-6, np.array([-20, -10, -10, 0])),
    "ITU Pedestrian A": (np.array([0, 110, 190, 410]) * 1e-9, np.array([0, -9.7, -19.2, -22.8])),
    "ITU Vehicular A": (np.array([0, 310, 710, 1090, 1730, 2510]) * 1e-9, np.array([0, -1, -9, -10, -15, -20])),
}
print(f"{'profile':20s} {'mean excess':>12s} {'rms spread':>11s} {'Bc (0.9)':>10s} {'Bc (0.5)':>10s}")
for name, (tau, pdb) in pdps.items():
    p = fs.delay_spread_parameters(tau, pdb, powers_in_db=True)
    s = p["rms_delay_spread"]
    print(f"{name:20s} {1e6*p['mean_excess_delay']:9.3f} us {1e6*s:8.3f} us "
          f"{1e-3*fs.coherence_bandwidth(s, 0.9):7.1f} kHz {1e-3*fs.coherence_bandwidth(s, 0.5):7.1f} kHz")

tau, pdb = pdps["Rappaport Ex. 5.4"]
p = fs.delay_spread_parameters(tau, pdb, powers_in_db=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9))
ax1.stem(1e6 * tau, pdb, bottom=-30, basefmt="k-")
tb, st = 1e6 * p["mean_excess_delay"], 1e6 * p["rms_delay_spread"]
ax1.axvline(tb, color="C3", lw=2, label=rf"$\bar\tau$ = {tb:.2f} $\mu$s")
ax1.axvspan(tb - st, tb + st, color="C3", alpha=0.15, label=rf"$\bar\tau\pm\sigma_\tau$ ($\sigma_\tau$ = {st:.2f} $\mu$s)")
ax1.set_xlabel(r"excess delay $\tau$ ($\mu$s)"); ax1.set_ylabel("relative power (dB)"); ax1.set_ylim(-30, 3)
ax1.set_title("Power delay profile (Rappaport Ex. 5.4)"); ax1.legend(fontsize=8, loc="upper left"); ax1.grid(alpha=0.3)

df = np.logspace(3, 7, 600)
for i, (name, (tau_i, pdb_i)) in enumerate(pdps.items()):
    s = fs.delay_spread_parameters(tau_i, pdb_i, powers_in_db=True)["rms_delay_spread"]
    ax2.semilogx(1e-3 * df, fs.frequency_correlation(tau_i, db_to_linear(pdb_i), df), color=f"C{i}", label=name)
    ax2.plot(1e-3 * fs.coherence_bandwidth(s, 0.5), 0.5, "v", color=f"C{i}", ms=9)
ax2.axhline(0.5, color="gray", ls=":"); ax2.axhline(0.9, color="gray", ls=":")
ax2.set_xlabel(r"frequency separation $\Delta f$ (kHz)"); ax2.set_ylabel(r"$|R(\Delta f)|$")
ax2.set_title(r"Frequency correlation; $\blacktriangledown$ = rule $1/(5\sigma_\tau)$")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The frequency correlation $|R(\Delta f)|$ is the Fourier transform of the PDP: the wider the PDP, the faster the
channel decorrelates in frequency. The $1/(5\sigma_\tau)$ rule is only an order-of-magnitude guide; the exact
0.5-correlation point depends on the *shape* of the PDP. The Rappaport and Pedestrian A profiles are dominated by one
strong path, so their correlation never drops to 0.5 at all.

**Interactive: flat vs. frequency-selective.** One random realization of the Vehicular A channel (each path an
independent Rayleigh gain), with all delays scaled by a factor. The shaded band is a signal of bandwidth $B_s$
centred on the carrier. When $B_s \ll B_c$ the channel is (nearly) constant across the band — flat fading; when
$B_s \gtrsim B_c$ several fades fall inside the band — frequency-selective fading.

In [ ]:
tau_va, pdb_va = pdps["ITU Vehicular A"]
g_va = fading.iid_rayleigh(len(tau_va), rng=5) * np.sqrt(db_to_linear(pdb_va) / db_to_linear(pdb_va).sum())

def show_freq_response(delay_scale=1.0, Bs_MHz=1.0):
    tau = tau_va * delay_scale
    s = fs.delay_spread_parameters(tau, pdb_va, powers_in_db=True)["rms_delay_spread"]
    Bc = fs.coherence_bandwidth(s, 0.5)
    f = np.linspace(-10e6, 10e6, 2001)
    H = fs.frequency_response(tau, g_va, f)
    inband = np.abs(f) <= Bs_MHz * 1e6 / 2
    HdB = 20 * np.log10(np.abs(H))
    ripple = HdB[inband].max() - HdB[inband].min()
    kind = fs.classify_channel(Bs_MHz * 1e6, Bc, 1 / (Bs_MHz * 1e6), 1.0)[0]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 2]})
    a1.stem(1e6 * tau, np.abs(g_va) ** 2, basefmt="k-")
    a1.set_xlabel(r"delay $\tau$ ($\mu$s)"); a1.set_ylabel(r"$|a_k|^2$ (this realization)")
    a1.set_title(rf"$\sigma_\tau$ = {1e9*s:.0f} ns"); a1.grid(alpha=0.3)
    a2.plot(1e-6 * f, HdB, "C0", lw=1.2)
    a2.axvspan(-Bs_MHz / 2, Bs_MHz / 2, color="C1", alpha=0.2, label=rf"signal band $B_s$ = {Bs_MHz:g} MHz")
    a2.set_ylim(-35, 12); a2.set_xlabel(r"frequency offset $f - f_c$ (MHz)"); a2.set_ylabel(r"$|H(f)|^2$ (dB)")
    a2.set_title(rf"$B_c\approx 1/(5\sigma_\tau)$ = {1e-3*Bc:.0f} kHz, $B_s/B_c$ = {Bs_MHz*1e6/Bc:.2f} "
                 rf"$\Rightarrow$ {kind} (in-band variation {ripple:.1f} dB)", fontsize=10)
    a2.legend(loc="lower left", fontsize=8); a2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_freq_response,
         delay_scale=FloatLogSlider(value=1.0, base=10, min=-1.5, max=1, step=0.05, description="delay ×"),
         Bs_MHz=FloatLogSlider(value=1.0, base=10, min=-2, max=np.log10(20), step=0.05, description="B_s (MHz)"));

**Coherence time.** The complex gain of a Clarke channel has time correlation $J_0(2\pi f_m\Delta t)$. Below, two
receivers moving at different speeds: the faster one ($f_m$ = 200 Hz) decorrelates in a quarter of the time.
The dashed lines mark $T_c = 0.423/f_m$.

In [ ]:
fs_hz = 10e3
t = np.arange(int(0.2 * fs_hz)) / fs_hz
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
lags = np.arange(0, 400)
for i, fm_i in enumerate([50.0, 200.0]):
    h = fading.rayleigh_sos(fm_i, t, rng=3)
    a1.plot(1e3 * t, 20 * np.log10(np.abs(h)), color=f"C{i}", lw=1, label=rf"$f_m$ = {fm_i:g} Hz")
    # empirical correlation averaged over several realizations
    hs = [fading.rayleigh_sos(fm_i, np.arange(8000) / fs_hz, rng=100 + s) for s in range(8)]
    rho_emp = np.mean([[np.vdot(x[:-l or None], x[l:]).real / (len(x) - l) for l in lags] for x in hs], axis=0)
    a2.plot(1e3 * lags / fs_hz, fs.time_correlation(lags / fs_hz, fm_i), color=f"C{i}", label=rf"$J_0$, $f_m$ = {fm_i:g} Hz")
    a2.plot(1e3 * lags[::12] / fs_hz, rho_emp[::12], "o", color=f"C{i}", mfc="none", ms=5, label="simulation")
    a2.axvline(1e3 * fs.coherence_time(fm_i), color=f"C{i}", ls="--", lw=1)
a1.set_ylim(-35, 10); a1.set_xlabel("time t (ms)"); a1.set_ylabel(r"$|h(t)|^2$ (dB)")
a1.set_title("Faster motion, faster fading"); a1.legend(fontsize=8); a1.grid(alpha=0.3)
a2.set_xlabel(r"time lag $\Delta t$ (ms)"); a2.set_ylabel(r"Re $\mathrm{E}[h^*(t)h(t+\Delta t)]$")
a2.set_title(r"Time correlation; dashed: $T_c = 0.423/f_m$"); a2.legend(fontsize=8); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Fading channel classification
Time dispersion and frequency dispersion lead to two *independent* classifications (Rappaport):

| based on delay spread | based on Doppler spread |
|---|---|
| **flat**: $B_s < B_c$, $\sigma_\tau < T_s$ | **slow**: $T_s < T_c$, low Doppler spread |
| **frequency selective**: $B_s > B_c$, $\sigma_\tau > T_s$ | **fast**: $T_s > T_c$, high Doppler spread |

Because $B_s\approx 1/T_s$, changing the symbol rate moves a system along a line of slope $-1$ on the map of
$T_s/T_c$ versus $B_s/B_c$. Low rates risk *fast* fading, high rates *frequency-selective* fading; in between there
is a window of slow, flat fading — as long as $B_c \gg 1/T_c$, i.e. $\sigma_\tau f_m\ll 1$, which holds for
practically every terrestrial channel. Markers show example signals ($B_s \approx 1/T_s$).

In [ ]:
systems = {"NB-IoT subcarrier (3.75 kHz)": 3.75e3, "LTE subcarrier (15 kHz)": 15e3,
           "GSM (270.8 kbaud)": 270.8e3, "Wi-Fi subcarrier (312.5 kHz)": 312.5e3,
           "LTE 20 MHz, single carrier": 18e6}

def show_classification(sigma_tau_us=1.0, speed_kmh=100.0, fc_GHz=2.0):
    Bc = fs.coherence_bandwidth(sigma_tau_us * 1e-6, 0.5)
    fm = speed_kmh / 3.6 * fc_GHz * 1e9 / 3e8
    Tc = fs.coherence_time(fm)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    lo, hi, ylo = 1e-4, 1e4, 1e-6
    ax.fill_between([lo, 1], ylo, 1, color="C2", alpha=0.15); ax.fill_between([1, hi], ylo, 1, color="C1", alpha=0.15)
    ax.fill_between([lo, 1], 1, hi, color="C0", alpha=0.15); ax.fill_between([1, hi], 1, hi, color="C3", alpha=0.15)
    kw = dict(ha="center", va="center", fontsize=11, fontweight="bold", color="0.25")
    ax.text(1e-3, 1e-5, "flat, slow", **kw); ax.text(3e2, 1e-2, "frequency selective,\nslow", **kw)
    ax.text(1e-2, 1e2, "flat, fast", **kw); ax.text(1e2, 1e2, "frequency selective,\nfast", **kw)
    Rs = np.logspace(0, 9, 400)                       # symbol rate 1 baud ... 1 Gbaud, B_s = R_s = 1/T_s
    ax.loglog(Rs / Bc, 1 / (Rs * Tc), "k-", lw=1.5, label="vary symbol rate $1/T_s$")
    for i, (name, Rs_i) in enumerate(systems.items()):
        ax.loglog(Rs_i / Bc, 1 / (Rs_i * Tc), "o", ms=9, color=f"C{i+4}", mec="k", label=name)
    ax.set_xlim(lo, hi); ax.set_ylim(ylo, hi)
    ax.set_xlabel(r"$B_s/B_c$  (equivalently $\approx 5\sigma_\tau/T_s$)"); ax.set_ylabel(r"$T_s/T_c$")
    ax.set_title(rf"$\sigma_\tau$ = {sigma_tau_us:g} $\mu$s ($B_c$ = {1e-3*Bc:.0f} kHz), $f_m$ = {fm:.0f} Hz ($T_c$ = {1e3*Tc:.2f} ms)")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1)); ax.grid(True, which="major", alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_classification,
         sigma_tau_us=FloatLogSlider(value=1.0, base=10, min=-2, max=1.3, step=0.05, description="σ_τ (μs)"),
         speed_kmh=FloatLogSlider(value=100, base=10, min=0, max=np.log10(500), step=0.05, description="v (km/h)"),
         fc_GHz=FloatSlider(value=2.0, min=0.5, max=40, step=0.5, description="f_c (GHz)"));

## 6. Modulation performance in fading channels
In a slow, flat fading channel the gain is constant over a symbol: $r(t) = \alpha(t)e^{-j\theta(t)}s(t)+n(t)$. The
instantaneous SNR is $\gamma = \alpha^2 E_b/N_0$ and the average error probability is

$$P_e = \int_0^\infty P_e(\gamma)\,p(\gamma)\,d\gamma,\qquad
\text{Rayleigh: } p(\gamma)=\frac{1}{\Gamma}e^{-\gamma/\Gamma},\quad \Gamma = \frac{E_b}{N_0}\overline{\alpha^2}.$$

For conditional error probabilities of the form $P_e(\gamma)=a\,Q(\sqrt{b\gamma})$ the integral is closed-form:

$$P_e = \frac{a}{2}\left(1-\sqrt{\frac{b\Gamma/2}{1+b\Gamma/2}}\right)\ \xrightarrow{\ \Gamma\to\infty\ }\ \frac{a}{2b\Gamma}.$$

BPSK and (Gray) QPSK: $a=1$, $b=2$; square $M$-QAM (nearest-neighbour approximation, $k=\log_2 M$):
$a=\tfrac{4}{k}(1-\tfrac{1}{\sqrt M})$, $b=\tfrac{3k}{M-1}$. Markers: Monte Carlo with coherent detection
($y/h$, perfect channel knowledge).

In [ ]:
def mc_ber(scheme, ebno_db, channel="rayleigh", K=0.0, n=100_000, rng=None):
    rng = np.random.default_rng(rng)
    pts = {"bpsk": mod.constellation("psk", 2), "qpsk": mod.constellation("psk", 4),
           "16qam": mod.constellation("qam", 16)}[scheme]
    k = mod.bits_per_symbol(pts)
    idx, s = mod.random_symbols(pts, n, rng)
    if channel == "awgn":
        h = np.ones(n)
    else:  # Ricean with E|h|^2 = 1 (K = 0: Rayleigh)
        h = np.sqrt(K / (K + 1)) + np.sqrt(1 / (K + 1)) * fading.iid_rayleigh(n, rng)
    y = mod.awgn(h * s, ebno_db + linear_to_db(k), rng)       # Es/N0 = k Eb/N0
    return mod.bit_error_rate(idx, mod.detect(y / h, pts), len(pts))

eb = np.arange(0, 31, 2.0)
eb_fine = np.linspace(0, 30, 200)
labels = {"bpsk": "BPSK", "qpsk": "QPSK", "16qam": "16-QAM"}
fig, ax = plt.subplots(figsize=(8, 5))
for i, sch in enumerate(labels):
    ax.semilogy(eb_fine, fs.ber_awgn(eb_fine, sch), color=f"C{i}", ls="--", lw=1.2)
    ax.semilogy(eb_fine, fs.ber_rayleigh(eb_fine, sch), color=f"C{i}", lw=1.5, label=f"{labels[sch]}" + (" (same BER as BPSK)" if sch == "qpsk" else ""))
    ber_a = [mc_ber(sch, e, "awgn", rng=j) for j, e in enumerate(eb[eb <= 14])]
    ber_r = [mc_ber(sch, e, "rayleigh", rng=j) for j, e in enumerate(eb)]
    ax.semilogy(eb[eb <= 14], ber_a, "s", color=f"C{i}", mfc="none", ms=5)
    ax.semilogy(eb, ber_r, "o", color=f"C{i}", mfc="none", ms=5)
ax.plot([], [], "k--", label="AWGN (theory)"); ax.plot([], [], "k-", label="Rayleigh (theory)")
ax.plot([], [], "ks", mfc="none", label="AWGN (sim.)"); ax.plot([], [], "ko", mfc="none", label="Rayleigh (sim.)")
ax.set_ylim(1e-5, 0.5); ax.set_xlabel(r"$E_b/N_0$ or mean $\Gamma$ (dB)"); ax.set_ylabel("bit error rate")
ax.set_title("BER in AWGN vs. flat Rayleigh fading"); ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, ncol=2, loc="lower left"); plt.show()

print("Eb/N0 needed for BER = 1e-3:")
for sch in labels:
    need_a = eb_fine[np.argmax(fs.ber_awgn(eb_fine, sch) < 1e-3)]
    g = np.linspace(0, 40, 4001); need_r = g[np.argmax(fs.ber_rayleigh(g, sch) < 1e-3)]
    print(f"  {labels[sch]:7s} AWGN {need_a:5.1f} dB   Rayleigh {need_r:5.1f} dB   fading penalty {need_r-need_a:5.1f} dB")

In AWGN the BER falls like a waterfall; in Rayleigh fading only **one decade per 10 dB** — the average is dominated
by the deep fades where $\gamma$ is small (compare the AFD/LCR discussion above). A LOS component helps: evaluating
the integral numerically with the Ricean $p(\gamma)$ (`fs.average_error_probability`) moves the curve from Rayleigh
($K=0$) towards AWGN ($K\to\infty$).

In [ ]:
pe_bpsk = lambda g: mod.qfunc(np.sqrt(2 * g))
eb2 = np.arange(0, 25, 1.0)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for i, K in enumerate([0, 2, 5, 10, 20]):
    ax.semilogy(eb2, fs.average_error_probability(pe_bpsk, eb2, K), color=f"C{i}", label=f"K = {K} (numerical integral)")
    ax.semilogy(eb2[::4], [mc_ber("bpsk", e, K=K, rng=5) for e in eb2[::4]], "o", color=f"C{i}", mfc="none", ms=5)
ax.semilogy(eb2, fs.ber_awgn(eb2), "k--", label="AWGN")
ax.plot([], [], "ko", mfc="none", label="simulation")
ax.set_ylim(1e-5, 0.5); ax.set_xlabel(r"mean $E_b/N_0$ (dB)"); ax.set_ylabel("bit error rate")
ax.set_title("BPSK in Ricean fading"); ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8); plt.show()

**Interactive: constellations with fading and noise** ($y[n] = h[n]x[n] + n[n]$, $h[n]=r[n]e^{-j\theta[n]}$).
Left: what the receiver sees. Right: after coherent compensation $y/h$ — the points are back on the grid, but the
noise is magnified whenever $|h|$ is small (red = symbol error).

In [ ]:
def show_constellation(scheme="16qam", ebno_db=15.0, K=0.0, awgn_only=False):
    rng = np.random.default_rng(1)
    pts = mod.constellation("qam", 16) if scheme == "16qam" else mod.constellation("psk", 4)
    n = 2000
    idx, s = mod.random_symbols(pts, n, rng)
    h = np.ones(n) if awgn_only else np.sqrt(K / (K + 1)) + np.sqrt(1 / (K + 1)) * fading.iid_rayleigh(n, rng)
    y = mod.awgn(h * s, ebno_db + linear_to_db(mod.bits_per_symbol(pts)), rng)
    z = y / h
    err = mod.detect(z, pts) != idx
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.8))
    a1.plot(y.real, y.imag, ".", ms=2, alpha=0.5, color="C0")
    a1.set_title("received $y = hx + n$")
    a2.plot(z[~err].real, z[~err].imag, ".", ms=2, alpha=0.5, color="C0", label="correct")
    a2.plot(z[err].real, z[err].imag, ".", ms=3, color="C3", label="symbol error")
    a2.set_title(f"equalized $y/h$: SER = {err.mean():.3f}")
    for a in (a1, a2):
        a.plot(pts.real, pts.imag, "k+", ms=10, mew=2)
        a.set_xlim(-2.5, 2.5); a.set_ylim(-2.5, 2.5); a.set_aspect("equal")
        a.set_xlabel("in-phase"); a.set_ylabel("quadrature"); a.grid(alpha=0.3)
    a2.legend(fontsize=8, loc="upper right", markerscale=4)
    ch = "AWGN" if awgn_only else ("Rayleigh" if K == 0 else f"Ricean K = {K:g}")
    fig.suptitle(f"{'16-QAM' if scheme == '16qam' else 'QPSK'}, {ch}, $E_b/N_0$ = {ebno_db:g} dB")
    plt.tight_layout(); plt.show()

interact(show_constellation, scheme=Dropdown(options=[("16-QAM", "16qam"), ("QPSK", "qpsk")], value="16qam"),
         ebno_db=FloatSlider(value=15, min=0, max=30, step=1, description="Eb/N0 (dB)"),
         K=FloatSlider(value=0, min=0, max=20, step=0.5, description="K"),
         awgn_only=Checkbox(value=False, description="AWGN only (no fading)"));

## 7. OFDM
A single carrier with $B_s > B_c$ suffers frequency-selective fading (ISI, complex equalizers). OFDM splits the band
into $N$ narrow sub-carriers so that each one is flat:

$$\Delta f = \frac{B}{N}\ll B_c\propto\frac{1}{\sigma_\tau}\qquad\Longleftrightarrow\qquad T_{\mathrm{sym}} = \frac{1}{\Delta f}\gg\sigma_\tau.$$

The transmitter maps $N$ symbols $X_k$ onto the sub-carriers with an IFFT and prepends a **cyclic prefix** (the last
$N_{\mathrm{CP}}$ samples). If $T_{\mathrm{CP}}\ge\tau_{\max}$ the linear channel convolution becomes circular, and
after the receiver FFT every sub-carrier sees a single complex gain:

$$Y_k = H_k X_k + N_k \quad\Rightarrow\quad \hat X_k = Y_k/H_k\ \ \text{(one-tap equalizer)}.$$

We use IEEE 802.11a/g numerology: $B$ = 20 MHz sampling, $N=64$, $\Delta f$ = 312.5 kHz, $T_{\mathrm{sym}}$ = 3.2 µs,
$N_{\mathrm{CP}}=16$ samples = 0.8 µs.

In [ ]:
N, NCP, FS = 64, 16, 20e6
rng = np.random.default_rng(0)
qpsk = mod.constellation("psk", 4)
X = mod.random_symbols(qpsk, 3 * N, rng)[1].reshape(3, N)
x = ofdm.modulate(X, NCP)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
tt = np.arange(x.size) / FS * 1e6
a1.plot(tt, x.real, "C0", lw=1)
for m in range(3):
    t0 = m * (N + NCP) / FS * 1e6
    a1.axvspan(t0, t0 + NCP / FS * 1e6, color="C1", alpha=0.25, label="cyclic prefix" if m == 0 else None)
    a1.plot(tt[m*(N+NCP)+N : m*(N+NCP)+N+NCP], x.real[m*(N+NCP)+N : m*(N+NCP)+N+NCP], "C1", lw=2,
            label="copied tail" if m == 0 else None)
a1.set_xlabel(r"time ($\mu$s)"); a1.set_ylabel("Re x[n]"); a1.set_title("Three OFDM symbols (QPSK), CP highlighted")
a1.legend(fontsize=8, loc="upper right"); a1.grid(alpha=0.3)

f = np.linspace(-4, 4, 1001)          # in units of Delta f
for k in range(-2, 3):
    a2.plot(f, np.sinc(f - k), lw=1.5, label=f"sub-carrier {k:+d}")
a2.plot(np.arange(-2, 3), np.ones(5), "k.", ms=8)
a2.axhline(0, color="k", lw=0.8)
a2.set_xlabel(r"frequency $(f-f_c)/\Delta f$"); a2.set_ylabel("sub-carrier spectrum (amplitude)")
a2.set_title("Orthogonality: each peak sits on the others' nulls"); a2.legend(fontsize=7); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("round trip without channel, max |X - X_hat| =", np.abs(ofdm.demodulate(x, N, NCP) - X).max())

**Through a multipath channel.** An exponential power delay profile with $\sigma_\tau$ = 100 ns (indoor, sampled at
50 ns) — much shorter than the CP. The channel is frequency selective across the 20 MHz band ($|H_k|$ varies by
tens of dB), yet each 312.5 kHz sub-carrier is flat, so one complex division per sub-carrier undoes it.

In [ ]:
def exp_pdp_channel(sigma_tau_s, n_taps, rng):
    # Rayleigh taps with an exponentially decaying PDP, unit total mean power
    tau = np.arange(n_taps) / FS
    p = np.exp(-tau / sigma_tau_s)
    return tau, p / p.sum(), fading.iid_rayleigh(n_taps, rng) * np.sqrt(p / p.sum())

tau, p, h = exp_pdp_channel(100e-9, 12, np.random.default_rng(3))
print(f"PDP: sigma_tau = {1e9*fs.delay_spread_parameters(tau, p)['rms_delay_spread']:.0f} ns,"
      f"  max excess delay = {1e9*tau[-1]:.0f} ns  <  T_CP = {1e9*NCP/FS:.0f} ns")
res = ofdm.simulate_link(mod.constellation("qam", 16), N, NCP, h, snr_db=25, n_sym=30, rng=1)
k = np.fft.fftshift(np.arange(N) - N // 2)   # FFT index -> signed sub-carrier number
order = np.argsort(k)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].plot(k[order], 20 * np.log10(np.abs(res["H"][order])), "C0.-")
ax[0].set_xlabel("sub-carrier k"); ax[0].set_ylabel(r"$|H_k|^2$ (dB)"); ax[0].set_title("Channel across the 20 MHz band")
Yd = res["Y"][:, res["data_idx"]].ravel()
ax[1].plot(Yd.real, Yd.imag, ".", ms=2, alpha=0.5); ax[1].set_title(r"before equalization: $Y_k$")
Xh = res["X_hat"].ravel()
ax[2].plot(Xh.real, Xh.imag, ".", ms=2, alpha=0.5); ax[2].set_title(rf"after one-tap: $Y_k/H_k$ (BER = {res['ber']:.4f})")
for a in ax[1:]:
    a.set_xlim(-3, 3); a.set_ylim(-3, 3); a.set_aspect("equal"); a.set_xlabel("in-phase"); a.set_ylabel("quadrature")
for a in ax:
    a.grid(alpha=0.3)
fig.suptitle("16-QAM OFDM, mean SNR 25 dB per sub-carrier"); plt.tight_layout(); plt.show()

**BER vs. SNR.** Averaged over many channel realizations, each sub-carrier gain $H_k$ is complex Gaussian, so
uncoded OFDM with perfect channel knowledge behaves exactly like **flat Rayleigh fading** — the curve from
section 6. OFDM removes ISI; it does not remove fading (that needs coding across sub-carriers or diversity,
Week 8). With pilot-based LS estimates (linear interpolation) two effects appear: at low SNR the noisy pilot
estimates cost a few dB, and at high SNR an **error floor** appears because the interpolation cannot follow $H_k$
between pilots — much worse for spacing $D=4$ than $D=2$ (the channel memory here is 11 samples, so $H_k$ wiggles
on a scale of $64/11\approx 6$ sub-carriers). ($E_b/N_0$ ignores the CP and pilot overhead.)

In [ ]:
snr_eb = np.arange(0, 31, 3.0)
n_ch = 60
ber_perf, ber_ls, ber_ls2 = [], [], []
rng = np.random.default_rng(10)
for e in snr_eb:
    es = e + linear_to_db(2)            # QPSK: Es/N0 = 2 Eb/N0
    bp, bl, bl2 = [], [], []
    for _ in range(n_ch):
        h = exp_pdp_channel(100e-9, 12, rng)[2]
        bp.append(ofdm.simulate_link(qpsk, N, NCP, h, es, n_sym=10, rng=rng)["ber"])
        bl.append(ofdm.simulate_link(qpsk, N, NCP, h, es, n_sym=10, pilot_spacing=4, rng=rng)["ber"])
        bl2.append(ofdm.simulate_link(qpsk, N, NCP, h, es, n_sym=10, pilot_spacing=2, rng=rng)["ber"])
    ber_perf.append(np.mean(bp)); ber_ls.append(np.mean(bl)); ber_ls2.append(np.mean(bl2))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.semilogy(eb_fine, fs.ber_awgn(eb_fine, "qpsk"), "k--", label="AWGN theory")
ax.semilogy(eb_fine, fs.ber_rayleigh(eb_fine, "qpsk"), "k-", label="flat Rayleigh theory")
ax.semilogy(snr_eb, ber_perf, "o", color="C0", mfc="none", label="OFDM sim., perfect $H_k$")
ax.semilogy(snr_eb, ber_ls2, "^-", color="C2", mfc="none", lw=0.8, label="OFDM sim., LS pilots D = 2")
ax.semilogy(snr_eb, ber_ls, "s-", color="C1", mfc="none", lw=0.8, label="OFDM sim., LS pilots D = 4")
ax.set_ylim(1e-5, 0.5); ax.set_xlabel(r"$E_b/N_0$ per sub-carrier (dB)"); ax.set_ylabel("bit error rate")
ax.set_title("QPSK-OFDM over a multipath Rayleigh channel"); ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8)
plt.show()

**Interactive: cyclic prefix vs. delay spread.** The channel here has taps up to $\tau_{\max}$ (power falling by
10 dB across the profile). When $N_{\mathrm{CP}}$ samples no longer cover the channel memory
($T_{\mathrm{CP}} < \tau_{\max}$), the previous symbol leaks into the FFT window (ISI) and the lost cyclicity causes
inter-carrier interference: the one-tap equalizer can no longer clean the constellation, even at 40 dB SNR.
Right panel: signal-to-interference-plus-noise ratio after equalization vs. CP length.

In [ ]:
def show_cp(n_cp=16, tau_max_samples=12):
    L = tau_max_samples + 1
    pdb = -10 * np.arange(L) / max(L - 1, 1)
    p = db_to_linear(pdb); p /= p.sum()
    h = fading.iid_rayleigh(L, np.random.default_rng(4)) * np.sqrt(p)
    qam16 = mod.constellation("qam", 16)
    res = ofdm.simulate_link(qam16, N, n_cp, h, 40.0, n_sym=20, rng=2)
    cps = np.arange(0, 33, 2)
    sinr = []
    for c in cps:
        r_c = ofdm.simulate_link(qam16, N, c, h, 40.0, n_sym=20, rng=2)
        sinr.append(-linear_to_db(np.mean(np.abs(r_c["X_hat"].ravel() - qam16[r_c["tx_idx"]]) ** 2)))
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    ax[0].stem(np.arange(L) / FS * 1e6, np.abs(h) ** 2, basefmt="k-")
    ax[0].axvspan(0, n_cp / FS * 1e6, color="C1", alpha=0.2, label=rf"$T_{{\mathrm{{CP}}}}$ = {1e3*n_cp/FS*1e6:.0f} ns")
    ax[0].set_xlim(-0.05, 33 / FS * 1e6); ax[0].set_xlabel(r"delay ($\mu$s)"); ax[0].set_ylabel(r"$|h[n]|^2$")
    ax[0].set_title(rf"channel: $\tau_{{\max}}$ = {1e3*tau_max_samples/FS*1e6:.0f} ns"); ax[0].legend(fontsize=8)
    z = res["X_hat"].ravel()
    ax[1].plot(z.real, z.imag, ".", ms=2, alpha=0.5); ax[1].plot(qam16.real, qam16.imag, "k+", ms=8)
    ax[1].set_xlim(-2, 2); ax[1].set_ylim(-2, 2); ax[1].set_aspect("equal")
    ok = "no ISI" if n_cp >= tau_max_samples else "ISI!"
    ax[1].set_title(f"equalized 16-QAM ({ok}), BER = {res['ber']:.4f}"); ax[1].set_xlabel("in-phase"); ax[1].set_ylabel("quadrature")
    ax[2].plot(cps, sinr, "C0o-", ms=4); ax[2].axvline(tau_max_samples, color="C3", ls="--", label=r"$N_{\mathrm{CP}} = \tau_{\max}/T_s$")
    ax[2].plot(n_cp, sinr[np.argmin(np.abs(cps - n_cp))] if n_cp in cps else np.nan, "C1*", ms=14)
    ax[2].set_xlabel(r"CP length $N_{\mathrm{CP}}$ (samples of 50 ns)"); ax[2].set_ylabel("SINR after equalizer (dB)")
    ax[2].set_title("CP must cover the channel memory"); ax[2].legend(fontsize=8)
    for a in ax: a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_cp, n_cp=IntSlider(value=16, min=0, max=32, step=1, description="N_CP"),
         tau_max_samples=IntSlider(value=12, min=0, max=32, step=1, description="τ_max (samples)"));

**Pilot-based channel estimation.** The receiver does not know $H_k$. Known pilot symbols $X_p$ are inserted on a
comb of sub-carriers (spacing $D$); the least-squares estimate at a pilot is simply $\hat H_p = Y_p/X_p$, and the
data sub-carriers are filled in by interpolation. The comb samples $H(f)$ every $D\,\Delta f$ Hz, so it must be
dense compared with the coherence bandwidth — by the sampling theorem (in frequency) $D\,\Delta f < 1/\tau_{\max}$,
i.e. $D < N/(\tau_{\max}/T_s)$.
Try a large pilot spacing with a long delay spread, and compare the interpolation methods.

In [ ]:
def show_pilots(spacing=8, snr_db=20.0, sigma_tau_ns=100.0, method="linear"):
    rng = np.random.default_rng(21)
    n_taps = min(int(np.ceil(5 * sigma_tau_ns * 1e-9 * FS)) + 1, NCP + 1)
    tau, p, h = exp_pdp_channel(sigma_tau_ns * 1e-9, n_taps, rng)
    res = ofdm.simulate_link(qpsk, N, NCP, h, snr_db, n_sym=1, pilot_spacing=spacing, method=method, rng=rng)
    H, Hh, pi = res["H"], res["H_hat"][0], res["pilot_idx"]
    mse = np.mean(np.abs(Hh - H) ** 2) / np.mean(np.abs(H) ** 2)
    Bc = fs.coherence_bandwidth(fs.delay_spread_parameters(tau, p)["rms_delay_spread"], 0.5)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    kk = np.arange(N)
    for a, part, lab in ((a1, np.real, "Re"), (a2, np.imag, "Im")):
        a.plot(kk, part(H), "k-", lw=1.5, label="true $H_k$")
        a.plot(kk, part(Hh), "C1--", lw=1.5, label=f"LS + {method} interpolation")
        a.plot(pi, part(res["Y"][0, pi]), "C0o", ms=6, label=r"pilot LS $Y_p/X_p$")
        a.set_xlabel("sub-carrier index k (FFT order)"); a.set_ylabel(f"{lab} $H_k$"); a.grid(alpha=0.3)
    a1.legend(fontsize=8)
    fig.suptitle(rf"pilot spacing D = {spacing} ({spacing*312.5:g} kHz), $B_c\approx$ {1e-3*Bc:.0f} kHz, "
                 rf"SNR = {snr_db:g} dB: normalized MSE = {linear_to_db(mse):.1f} dB, BER = {res['ber']:.4f}")
    plt.tight_layout(); plt.show()

interact(show_pilots, spacing=IntSlider(value=8, min=2, max=16, step=1, description="pilot spacing D"),
         snr_db=FloatSlider(value=20, min=0, max=40, step=1, description="SNR (dB)"),
         sigma_tau_ns=FloatSlider(value=100, min=10, max=300, step=10, description="σ_τ (ns)"),
         method=Dropdown(options=["linear", "cubic", "nearest"], value="linear"));

Finally, the estimation error averaged over many channels, as a function of pilot spacing: at high SNR the error is
set by interpolation (it explodes once $D\,\Delta f$ approaches the coherence bandwidth); at low SNR by noise.

In [ ]:
spacings = np.array([2, 3, 4, 6, 8, 12, 16])
fig, ax = plt.subplots(figsize=(7.5, 4.3))
rng = np.random.default_rng(30)
chans = [exp_pdp_channel(150e-9, NCP + 1, rng)[2] for _ in range(40)]
for i, method in enumerate(["nearest", "linear", "cubic"]):
    for snr, ls in ((10, ":"), (30, "-")):
        m = []
        for D in spacings:
            e = [np.mean(np.abs(r["H_hat"] - r["H"]) ** 2) for r in
                 (ofdm.simulate_link(qpsk, N, NCP, h, snr, n_sym=2, pilot_spacing=D, method=method, rng=rng) for h in chans)]
            m.append(linear_to_db(np.mean(e)))
        ax.plot(spacings, m, ls, marker="o", ms=4, color=f"C{i}", label=f"{method}, SNR {snr} dB")
ax.set_xlabel("pilot spacing D (sub-carriers)"); ax.set_ylabel(r"MSE of $\hat H_k$ (dB)")
ax.set_title(r"LS channel estimation error ($\sigma_\tau$ = 150 ns, $B_c\approx$ 1.3 MHz $\approx$ 4.3 $\Delta f$)")
ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2); plt.show()

## Try it yourself
* A mobile at 90 km/h receives a 1.9 GHz carrier. Using section 3, what is the average fade duration at
  $\rho = -20$ dB? If a voice codec frame is 20 ms long, would a typical fade corrupt one bit or a whole burst?
  How does this change for a pedestrian at 5 km/h?
* In the interactive LCR plot, why does a Ricean channel with $K=10$ have a *higher* LCR than Rayleigh near
  $\rho = 0$ dB but a far lower one at $-20$ dB? What happens to the agreement with theory when the LOS path has a
  Doppler shift, and which assumption of the derivation is violated?
* Using the classification map, find the range of symbol rates for which the ITU Vehicular A channel
  ($\sigma_\tau\approx 0.37~\mu$s) at 120 km/h and 3.5 GHz is slow *and* flat. Where does a 30 kHz 5G NR
  sub-carrier fall?
* In the OFDM CP demo, the SINR does not jump from bad to perfect exactly at $N_{\mathrm{CP}} = \tau_{\max}/T_s$ but
  improves gradually. Why? What is the throughput cost of choosing $N_{\mathrm{CP}}=16$ vs. $32$ with $N=64$, and why
  do LTE/5G use a much larger $N$?